# Ordered Logistic Regression Results: Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll access the dataset via its Croissant schema, examine metadata and available record sets, extract structured data, and perform basic exploratory analysis.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

In [ ]:
# Ensure `mlcroissant` is installed (uncomment if running for the first time)
!pip install mlcroissant

## 1. Data Loading
Let's load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
if hasattr(metadata, 'keywords'):
    print(f"Keywords: {metadata.keywords}\n")

## 2. Data Overview
Let's review the available record sets, fields, and their `@id` references.

For each record set in the dataset, we will print its `@id`, name, and the fields (using their `@id`s).

In [ ]:
# Retrieve available record set @ids from metadata
record_sets = dataset.metadata.record_set
if not record_sets:
    print("No record sets defined in the metadata.")
else:
    if not isinstance(record_sets, list):
        record_sets = [record_sets]
    for record_set in record_sets:
        print(f"Record Set @id: {record_set['@id']}")
        print(f"  Name: {record_set.get('name', 'N/A')}")
        # Show field @ids
        field_ids = [field['@id'] for field in record_set.get('field', [])] if record_set.get('field') else []
        print(f"  Field @ids: {field_ids if field_ids else 'None found'}\n")

## 3. Data Extraction
We'll extract data from each available record set into a pandas DataFrame for analysis, referencing record set and field `@id`s as discovered above.

In [ ]:
# We'll use the @ids from the previous cell. If none are available, we continue with an example.

dataframes = {}
loaded_recordsets = []
if not record_sets:
    print("Skipping extraction: no record sets defined.")
else:
    for record_set in record_sets:
        rec_id = record_set['@id']
        try:
            records = list(dataset.records(record_set=rec_id))
            df = pd.DataFrame(records)
            dataframes[rec_id] = df
            loaded_recordsets.append(rec_id)
            print(f"Record Set '{rec_id}' loaded with {len(df)} records and {len(df.columns)} columns.")
        except Exception as e:
            print(f"Could not load record set {rec_id}: {e}")

if loaded_recordsets:
    chosen_recordset = loaded_recordsets[0]
    print(f"\nAvailable columns in record set '{chosen_recordset}':")
    print(dataframes[chosen_recordset].columns.tolist())
    dataframes[chosen_recordset].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform some simple data processing and analysis, such as filtering records, normalizing numeric fields, and grouping data. All columns/fields are referenced by their `@id`s as per Croissant conventions.

> **Note:** Adjust `<numeric_field_id>` and `<group_field_id>` as appropriate for your record set's schema.

In [ ]:
# Choose the main record set (first loaded if available)
if loaded_recordsets:
    record_set_id = chosen_recordset
    df = dataframes[record_set_id]
    
    # Display available columns to assist user selection
    print("Columns (available field @ids):")
    print(list(df.columns))
    
    # For illustration, let's try to pick a numeric field
    import numpy as np
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df[[numeric_field]].head())
        
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean())
            / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try grouping by a categorical field, if any
        cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field = cat_cols[0] if cat_cols else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No record set loaded: skipping EDA.")

## 5. Visualization
Now, let's visualize distributions or relationships found in the dataset.

For this example, a histogram of the selected numeric field and a bar plot of the grouped means are shown.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if loaded_recordsets and 'numeric_field' in locals():
    # Histogram of normalized values
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f'{numeric_field}_normalized'], bins=20, kde=True)
    plt.title(f'Normalized Distribution of {numeric_field}')
    plt.xlabel(f'{numeric_field}_normalized')
    plt.show()
        
    # Bar plot if grouping was done
    if 'grouped_df' in locals() and 'group_field' in locals():
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field selected or no data available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. All structural references (record sets, fields) are handled using their `@id` identifiers as recommended by the Croissant specification. This workflow enables more reproducible and transparent data science with FAIR datasets.

You may continue refining your exploratory analysis and visualizations by referencing specific fields or record sets discovered during step 2. For further insight, see the dataset's official documentation or the Croissant schema definition.